In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps02.ipynb")

# PS2 — Expression Dynamics and the Cost of Speed
### BioE 147/247 · Fall 2026

**Out:** Thursday, September 10 · **Due:** Thursday, September 17, 11:59 pm
**Covers:** Session 5 · **31 points** · **Question 5** is required for **BioE 247** (36 total) and is extra credit for **BioE 147** (up to +5)

---

A short set, on purpose. It is one first-order differential equation, examined
properly.

Everything here uses the numbers from Thursday: *E. coli* dividing every
30 minutes, and the four ssrA tag variants from Andersen et al. 1998.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard together,
then write your own solution and your own code. Record who you worked with
below.

**If you used an LLM**, say so briefly and say what for. The conditions are that
you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — limits, monotonicity,
scaling. A green visible check means "not obviously broken", not "right".

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q2"
AI_USE = ""          # e.g. "used an LLM to check my algebra in Q1"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


---
## Question 1 — Two removal processes, one rate

*T10, T12.*

A protein is made at a constant rate α per cell per minute, degraded with rate
constant γ, and diluted by growth with rate constant μ:

$$\frac{dp}{dt} = \alpha - (\gamma + \mu)\,p$$

For a host with doubling time $T_d$, $\mu = \ln 2 / T_d$.

**Q1a.** Write `mu_from_doubling(T_d)` returning the dilution rate constant
for a doubling time in minutes.

In [ ]:
def mu_from_doubling(T_d):
    """Dilution rate constant (per min) for a doubling time in minutes."""
    ...


print(f"T_d = 30 min -> mu = {mu_from_doubling(30.0):.4f} /min")

In [ ]:
grader.check("q1a")

**Q1b.** Write `half_time(t_half_deg, T_d)` returning the **response time**
$t_{1/2} = \ln 2/(\gamma + \mu)$ in minutes.

`t_half_deg` is the degradation half-life in minutes. For an untagged protein
with no measurable degradation, pass `np.inf` — and note that you then need no
special case at all, because `ln 2 / ∞ = 0`.

In [ ]:
def half_time(t_half_deg, T_d):
    """Response time (min): ln2/(gamma+mu). t_half_deg=np.inf means gamma=0."""
    ...


for tag, th in [("no tag", np.inf), ("ASV", 110.), ("AAV", 60.), ("LAA", 40.)]:
    print(f"{tag:>7}: t_half = {half_time(th, 30.0):5.1f} min")

In [ ]:
grader.check("q1b")

**Q1c.** Write `steady_state_ratio(t_half_deg, T_d)` returning the
steady-state level **relative to the same construct with no degradation tag**,
at fixed α.

As in Q1b, `t_half_deg = np.inf` means no tag.

Then run the cell below it, which prints the table from Thursday.

In [ ]:
def steady_state_ratio(t_half_deg, T_d):
    """p* with the tag over p* untagged, at the same alpha. inf = no tag."""
    ...


print(f"{'tag':>8} {'t_half':>8} {'p*/p*_none':>11}")
for tag, th in [("no tag", np.inf), ("ASV", 110.), ("AAV", 60.), ("LAA", 40.)]:
    print(f"{tag:>8} {half_time(th, 30.0):7.1f}m {steady_state_ratio(th, 30.0):11.2f}")

In [ ]:
grader.check("q1c")

<!-- BEGIN QUESTION -->

**Q1d.** *(written)* Your test in Q1c checked that

$$\frac{p^*_{\text{tagged}}}{p^*_{\text{untagged}}}
= \frac{t_{1/2,\text{tagged}}}{t_{1/2,\text{untagged}}}$$

Explain in two or three sentences **why that identity must hold**, from the
model rather than from the arithmetic. Then say what it means for a designer who
wants a circuit that is both fast *and* strongly expressed.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — A specification you were handed

*T11.*

You are asked for a reporter that reaches half its final level within
**12 minutes** of induction, in a host that divides every **25 minutes**. You
may use any of the four Andersen tags: ASV (110 min), AAV (60 min),
LVA or LAA (40 min), or none.

**Q2a.** Write `required_deg_half_life(target_t_half, T_d)` returning the
degradation half-life you would need, in minutes, to hit a target response time
in that host — or `np.inf` if dilution alone is already fast enough, or
`np.nan` if no amount of degradation can reach the target.

(The last case cannot happen for a positive target, but say so in code rather
than assuming it.)

In [ ]:
def required_deg_half_life(target_t_half, T_d):
    """Degradation half-life needed to reach target_t_half in this host."""
    ...


need = required_deg_half_life(12.0, 25.0)
print(f"need a degradation half-life of {need:.1f} min")

In [ ]:
grader.check("q2a")

<!-- BEGIN QUESTION -->

**Q2b.** *(written)* Using your answer to Q2a:

1. Can any tag in Andersen et al. meet this specification? Give the number that
   decides it.
2. Report the response time and the relative steady-state level you would
   actually get with the **best** available tag.
3. The specification cannot be met by choosing a tag. Give **two** other things
   a designer could change, and for each say what it costs. One of them should
   not involve changing the circuit at all.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — Getting a rate out of data

*T10, T11.*

`posb.data.decay_timecourse(tag)` returns `(t_min, fluorescence)` for a
translation-shutoff experiment: at $t=0$ production is stopped, and what is left
decays. Data are noisy, as data are.

**Q3a.** Write `fit_removal_rate(t, y)` returning the removal rate
constant, per minute, from a semi-log fit.

Fit $\ln y$ against $t$ by least squares and return the rate constant as a
**positive** number. Do not use a nonlinear solver; one `np.polyfit` is
enough.

In [ ]:
def fit_removal_rate(t, y):
    """Removal rate constant (per min, positive) from a semi-log fit."""
    ...


t, y = posb.data.decay_timecourse("LAA")
k = fit_removal_rate(t, y)
print(f"k = {k:.4f} /min  ->  half-life {np.log(2)/k:.1f} min")

In [ ]:
grader.check("q3a")

<!-- BEGIN QUESTION -->

**Q3b.** *(written)* The experiment above was done in **growing** cells.

1. Is the rate constant you fitted γ, or γ + μ? Say how you know from the
   description of the experiment, not from the number.
2. Andersen et al. measured their half-lives after a **medium downshift**, which
   arrests growth. Which quantity did *they* measure, and why does that make
   their numbers safe to add to a μ of your own?
3. A paper reports "the half-life of our reporter is 45 minutes" and says
   nothing else. What is the one question you must ask before you use that
   number?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 4 — Two routes to the same curve

*T10, T12.*

Everything so far came from one closed-form solution. That solution is only
worth trusting if something independent agrees with it, so here you integrate
the differential equation numerically and compare.

This is the same move as PS1 Q3, and it is the one you will need in PS4, where
the system is two-dimensional and there is no closed form to fall back on.

**Q4a.** Write `simulate(t, alpha, t_half_deg, T_d)` which integrates

$$\frac{dp}{dt} = \alpha - (\gamma + \mu)\,p, \qquad p(0) = 0$$

**numerically**, using `solve_ivp`, and returns $p$ evaluated at the times in
`t`. Use `t_eval=t` so the output lines up with the input.

Then write `closed_form(t, alpha, t_half_deg, T_d)` returning the analytic
solution you derived in class, and run the comparison cell.

In [ ]:
def simulate(t, alpha, t_half_deg, T_d):
    """Numerical solution of dp/dt = alpha - (gamma+mu) p, p(0) = 0."""
    ...


def closed_form(t, alpha, t_half_deg, T_d):
    """The analytic solution: (alpha/k)(1 - exp(-k t))."""
    ...


tt = np.linspace(0, 90, 400)
num = simulate(tt, 1.0, 40.0, 30.0)
ana = closed_form(tt, 1.0, 40.0, 30.0)
worst = np.max(np.abs(num - ana)) / ana[-1]
print(f"largest disagreement: {worst:.2e} of the final level")
print(f"half-completion, numerical: {np.interp(num[-1]/2, num, tt):.2f} min")
print(f"Q1b formula said:          {half_time(40.0, 30.0):.2f} min")

In [ ]:
grader.check("q4a")

<!-- BEGIN QUESTION -->

**Q4b.** *(written)* Your comparison cell prints a largest disagreement that is
very small — but not zero.

1. What is that residual? Say whether it is a modelling error or something else,
   and what would make it smaller.
2. Suppose it had come out at 0.05 instead. Name **two** distinct things that
   could be wrong, one in each of the two routes.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 5 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 5 of your 36
points.

**BioE 147** — optional, worth up to 5 points of extra credit on top of
31. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

<!-- BEGIN QUESTION -->

**Q5.** Everything above assumed production switches on as a step: α is
zero, then constant. Induction is not like that — inducer has to enter the cell
and the promoter responds over some time.

Model this by letting production ramp:

$$\frac{dp}{dt} = \alpha\left(1 - e^{-t/\tau_{\text{ind}}}\right)
- (\gamma + \mu)\,p, \qquad p(0) = 0$$

1. Solve for $p(t)$ analytically. (It is linear and first-order; an integrating
   factor is enough.)
2. Show that as $\tau_{\text{ind}} \to 0$ your solution reduces to the step
   response of Q4a.
3. The response time is no longer $\ln2/(\gamma+\mu)$. Without solving for it
   exactly, say whether it is longer or shorter, and give the condition on
   $\tau_{\text{ind}}$ under which the Q1b formula is still a good description.
   State that condition as a comparison between two timescales, in the style of
   session 4.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

